# Smart MCQ Solver Challenge

## TF-IDF Baseline Model

In this notebook we build our first baseline model using:

- TF-IDF
- Cosine Similarity
- mAP@3 Evaluation

This baseline will serve as a reference for later transformer-based models.

## Importing Libraries and Loading Data

In [9]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
train_pairs = pd.read_csv("train_pairs.csv")
test = pd.read_csv("test.csv")

## Model

In [11]:
train_pairs["text"] = (
    "Question: "
    + train_pairs["prompt"]
    + " Answer: "
    + train_pairs["option"]
)

In [12]:
train_pairs[["prompt", "option", "text"]].head()

,prompt,option,text
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Question: Pick the best possible answer: What ...
1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,Question: Pick the best possible answer: What ...
2,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,Question: Pick the best possible answer: What ...
3,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,Question: Pick the best possible answer: What ...
4,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,Question: Pick the best possible answer: What ...


In [13]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

train_vectors = vectorizer.fit_transform(train_pairs["text"])

print(train_vectors.shape)
print(train_vectors)

(10000, 11313)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 448969 stored elements and shape (10000, 11313)>
  Coords	Values
  (0, 8254)	0.017042227337644315
  (0, 7549)	0.048478789053211534
  (0, 1268)	0.048478789053211534
  (0, 7742)	0.04786477904929378
  (0, 479)	0.03408445467528863
  (0, 5916)	0.21081537464798983
  (0, 4669)	0.21081537464798983
  (0, 11081)	0.0922776932096893
  (0, 8691)	0.18045600511730012
  (0, 10421)	0.1857217118930594
  (0, 4827)	0.10540768732399491
  (0, 3689)	0.08522739862244065
  (0, 5657)	0.04498264861321846
  (0, 7140)	0.04498264861321846
  (0, 1257)	0.10913463840312532
  (0, 4831)	0.11391160821647094
  (0, 3682)	0.09329327527773697
  (0, 2175)	0.1257243800460207
  (0, 5020)	0.10209883638692116
  (0, 3092)	0.07211511552022389
  (0, 2584)	0.0838847969922087
  (0, 1237)	0.11915183237242298
  (0, 3419)	0.13166030191296305
  (0, 7378)	0.10260759737162035
  (0, 5241)	0.19952124736621718
  :	:
  (9999, 11181)	0.11509441910188335
  (9999, 425)	0.

In [14]:
question_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

# Fit on both questions and options
question_vectorizer.fit(
    pd.concat([
        train_pairs["prompt"],
        train_pairs["option"]
    ])
)

question_vectors = question_vectorizer.transform(train_pairs["prompt"])
option_vectors = question_vectorizer.transform(train_pairs["option"])

In [15]:
similarity_scores = cosine_similarity(
    question_vectors,
    option_vectors
)

In [16]:
train_pairs["similarity"] = similarity_scores.diagonal()

In [17]:
train_pairs[["prompt", "option", "similarity"]].head(10)

,prompt,option,similarity
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,0.125362
1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,0.139390
2,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,0.280271
3,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,0.350629
4,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,0.109391
5,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,0.448031
6,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,0.419393
7,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,0.467901
8,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,0.399698
9,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,0.350763


In [18]:
train_pairs["rank"] = (
    train_pairs
    .groupby("id")["similarity"]
    .rank(method="first", ascending=False)
)

In [ ]:
top3 = (
    train_pairs
    .sort_values(["id", "similarity"], ascending=[True, False])
    .groupby("id")
    .head(3)
)

top3.head()

In [22]:
top3_predictions = (
    train_pairs
    .sort_values(["id", "similarity"], ascending=[True, False])
    .groupby("id")["option_id"]
    .apply(list)
    .apply(lambda x: x[:3])
)

top3_predictions.head()

,option_id
id,
1,"[D, C, B]"
2,"[C, A, B]"
3,"[A, B, C]"
4,"[D, C, B]"
5,"[D, E, B]"


In [23]:
ground_truth = (
    train_pairs[train_pairs["label"] == 1]
    .set_index("id")["option_id"]
)

ground_truth.head()

,option_id
id,
1,B
2,A
3,C
4,B
5,A


In [24]:
def map_at_3(actual, predicted):
    score = 0.0

    for qid in actual.index:

        true_answer = actual[qid]
        predictions = predicted[qid]

        if true_answer in predictions:
            rank = predictions.index(true_answer) + 1
            score += 1 / rank

    return score / len(actual)

In [25]:
score = map_at_3(ground_truth, top3_predictions)

print(f"MAP@3 Score: {score:.4f}")

MAP@3 Score: 0.3331
